(1) lips / benchmark / dataset import: 대회에서 제공한 AirfRRANS 데이터셋과 점수 계산용 벤치마크 불러오는 필수 코드(삭제,수정X)

In [1]:
from lips import get_root_path
from lips.benchmark.airfransBenchmark import AirfRANSBenchmark
from lips.dataset.airfransDataSet import download_data
from lips.dataset.scaler.standard_scaler_iterative import StandardScalerIterative

(2) 기본 라이브러리 import: 딥러닝 모델을 만들고 학습시키기 위한 기본 도구들(점수 자체 영향X, 모델 구조 바꾸면 간접적 영향)

In [2]:
import os
import time
import random
import math 

from tqdm import tqdm
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.nn import BatchNorm1d, Identity
from torch.nn import Linear


(3) 경로/설정 관련 코드: 데이터 위치+평가설정 지정(점수 계산에 필요)

In [3]:
LIPS_PATH = get_root_path()
DIRECTORY_NAME = 'Dataset'
BENCHMARK_NAME = "DEFAULT"
LOG_PATH = LIPS_PATH + "lips_logs.log"
BENCH_CONFIG_PATH = os.path.join("airfoilConfigurations","benchmarks","confAirfoil.ini") 

directory_name='Dataset'

(4) AirfRANSBenchmark 생성&데이터 로드: 데이터셋 로드+평가 시스템 초기화(수정X)

In [4]:
benchmark=AirfRANSBenchmark(benchmark_path = DIRECTORY_NAME,
                            config_path = BENCH_CONFIG_PATH,
                            benchmark_name = BENCHMARK_NAME,
                            log_path = LOG_PATH)
benchmark.load(path=DIRECTORY_NAME)

Loading dataset (task: scarce, split: train):   0%|          | 0/200 [00:00<?, ?it/s]

Loading dataset (task: reynolds, split: test): 100%|██████████| 496/496 [02:40<00:00,  3.10it/s]


(5) torc_geometric import: 그래프 신경망(GNN)+GAT 사용 위한 라이브러리(성능에 큰 영향)

In [5]:
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
print("torch_geometric OK")


torch_geometric OK


In [6]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv

from torch_geometric.utils import to_undirected


(6) build_knn_edge_index 함수: 각 노드를 좌표 기준으로 k-최근접 이웃 그래프로 연결(k: 점수에 매우 큰 영향)

In [7]:
def build_knn_edge_index(pos: torch.Tensor, k: int, make_undirected: bool = True) -> torch.Tensor:
    """
    pos: (N, 2) float tensor
    return edge_index: (2, E)
    """
    try:
        from torch_geometric.nn import knn_graph
        edge_index = knn_graph(pos, k=k, loop=False)
    except Exception:
        # fallback: sklearn NearestNeighbors
        from sklearn.neighbors import NearestNeighbors
        pos_np = pos.detach().cpu().numpy()
        nbrs = NearestNeighbors(n_neighbors=k+1, algorithm="auto").fit(pos_np)
        _, indices = nbrs.kneighbors(pos_np)  # (N, k+1) includes self
        src = np.repeat(np.arange(pos_np.shape[0]), k)
        dst = indices[:, 1:].reshape(-1)  # drop self
        edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long, device=pos.device)

    if make_undirected:
        edge_index = to_undirected(edge_index)

    return edge_index


(7) subsample_and_rebuild_graph 함수: 전체 CFD 메시에서 일부 노드만 뽑아 새로운 그래프 생성 함수(surf 점과 vol 점을 비율로 뽑아 다시 그래프 만듦)

In [8]:
def subsample_and_rebuild_graph(data: Data, n_sub: int, surf_ratio: float, k: int) -> Data:
    """
    data.pos: (N,2), data.x: (N,7), data.y: (N,4), data.surf: (N,)
    returns a NEW Data with:
      - subsampled nodes
      - rebuilt edge_index on the subsampled pos
    """
    N = data.x.size(0)
    n_sub = min(n_sub, N)

    surf_idx = torch.where(data.surf)[0]
    vol_idx  = torch.where(~data.surf)[0]

    n_surf = int(n_sub * surf_ratio)
    n_vol  = n_sub - n_surf

    # edge case: surface/volume가 너무 적을 때 자동 보정
    n_surf = min(n_surf, surf_idx.numel())
    n_vol  = min(n_vol, vol_idx.numel())
    if n_surf + n_vol < n_sub:
        # 부족분을 가능한 쪽에서 채움
        remain = n_sub - (n_surf + n_vol)
        if surf_idx.numel() - n_surf >= remain:
            n_surf += remain
        else:
            n_vol += remain

    # 랜덤 샘플링
    if n_surf > 0:
        perm_s = torch.randperm(surf_idx.numel(), device=surf_idx.device)[:n_surf]
        pick_s = surf_idx[perm_s]
    else:
        pick_s = torch.empty(0, dtype=torch.long, device=data.x.device)

    if n_vol > 0:
        perm_v = torch.randperm(vol_idx.numel(), device=vol_idx.device)[:n_vol]
        pick_v = vol_idx[perm_v]
    else:
        pick_v = torch.empty(0, dtype=torch.long, device=data.x.device)

    idx = torch.cat([pick_s, pick_v], dim=0)
    idx = idx[torch.randperm(idx.numel(), device=idx.device)]  # 섞기

    pos = data.pos[idx]
    x   = data.x[idx]
    y   = data.y[idx]
    surf = data.surf[idx]

    edge_index = build_knn_edge_index(pos, k=k, make_undirected=True)

    return Data(pos=pos, x=x, y=y, surf=surf.bool(), edge_index=edge_index)


(8) 하나의 CFD 시뮬레이션 그래프에서 각 노드의 물리량 예측 GAT 기반 신경망

In [9]:
class GATSurrogate(nn.Module):
    def __init__(self, in_dim=7, hidden_dim=128, out_dim=4,
                 k=12, heads=32, num_layers=3, dropout=0.1):
        super().__init__()
        assert hidden_dim == 128, "이 초기 버전은 hidden_dim=128 기준으로 맞춰져 있어요."

        self.dropout = dropout

        # encoder (node-wise)
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # GAT layers
        # layer 1~(L-1): concat=True로 head별 표현을 살림 (16*4=64)
        self.gat1 = GATConv(hidden_dim, 4, heads=heads, concat=True, dropout=dropout)
        self.gat2 = GATConv(128,        4, heads=heads, concat=True, dropout=dropout)

        # last layer: concat=False로 head를 평균/합쳐서 안정적으로 hidden_dim 유지
        self.gat3 = GATConv(128, 128, heads=heads, concat=False, dropout=dropout)

        # decoder (node-wise)
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, data: Data):
        x, edge_index = data.x, data.edge_index

        x = self.encoder(x)                      # (N,7) -> (N,64)

        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat1(x, edge_index))      # (N,64) -> (N,64)

        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat2(x, edge_index))      # (N,64) -> (N,64)

        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat3(x, edge_index))      # (N,64) -> (N,64)

        out = self.decoder(x)                    # (N,64) -> (N,4)
        return out


(9) 한 epoch 동안 여러 시뮬레이션을 subsampling해서 GAT 학습시키는 전체 학습 루프

In [10]:
def train_one_epoch(device, model, loader, optimizer, scheduler, criterion="MSE_weighted", surf_reg=1.0):
    model.train()

    loss_criterion = nn.MSELoss(reduction='none')
    total_loss_acc = 0.0
    it = 0

    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()

        out = model(data)
        targets = data.y

        loss_per_var = loss_criterion(out, targets).mean(dim=0)
        loss = loss_per_var.mean()

        # surface/volume 분리
        loss_surf_var = loss_criterion(out[data.surf, :], targets[data.surf, :]).mean(dim=0) if data.surf.any() else loss_per_var
        loss_vol_var  = loss_criterion(out[~data.surf, :], targets[~data.surf, :]).mean(dim=0) if (~data.surf).any() else loss_per_var
        loss_surf = loss_surf_var.mean()
        loss_vol  = loss_vol_var.mean()

        if criterion == "MSE_weighted":
            (loss_vol + surf_reg * loss_surf).backward()
            step_loss = (loss_vol + surf_reg * loss_surf).item()
        else:
            loss.backward()
            step_loss = loss.item()

        optimizer.step()
        scheduler.step()

        total_loss_acc += step_loss
        it += 1

    return total_loss_acc / max(it, 1)


def global_train(device, base_train_loader, model, hparams, criterion="MSE_weighted"):
    """
    base_train_loader: DataLoader(batch_size=1) where each item is ONE simulation Data (full nodes)
    매 epoch마다:
      - 각 simulation에서 subsample
      - subsample된 pos로 k-NN 재생성
      - 그 샘플들로 mini-batch 학습
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hparams["lr"])

    # scheduler: OneCycleLR (원 코드 느낌 유지)
    steps_per_epoch = max(len(base_train_loader) // hparams["batch_size"], 1)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=hparams["lr"],
        total_steps=steps_per_epoch * hparams["nb_epochs"]
    )

    for epoch in range(hparams["nb_epochs"]):
        sampled_list = []
        for data in base_train_loader:
            data = data.clone()

            # subsample + edge rebuild
            sampled = subsample_and_rebuild_graph(
                data=data,
                n_sub=hparams["subsampling"],
                surf_ratio=hparams["surf_ratio"],
                k=hparams["k"]
            )
            sampled_list.append(sampled)

        train_loader = DataLoader(sampled_list, batch_size=hparams["batch_size"], shuffle=True)

        avg_loss = train_one_epoch(
            device=device,
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            criterion=criterion,
            surf_reg=hparams.get("surf_reg", 1.0)
        )

        if (epoch + 1) % 10 == 0:
            print(f"[Epoch {epoch+1}/{hparams['nb_epochs']}] loss={avg_loss:.6f}")

    return model


(10) Lips benchmark가 요구하는 입출력 규격을 만족시키는 래퍼

In [11]:
class AugmentedSimulator():
    def __init__(self, benchmark, **kwargs):
        self.name = "AirfRANSSubmission"
        chunk_sizes = benchmark.train_dataset.get_simulations_sizes()
        scalerParams = {"chunk_sizes": chunk_sizes}
        self.scaler = StandardScalerIterative(**scalerParams)

        self.hparams = kwargs
        self.device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        print("Using", self.device)

        self.model = GATSurrogate(
            in_dim=7,
            hidden_dim=kwargs["hidden_dim"],
            out_dim=4,
            heads=kwargs["heads"],
            dropout=kwargs["dropout"]
        )
        

    def predict(self, dataset, **kwargs):
        """
        LIPS Benchmark가 호출하는 필수 메서드.
        반환은 dataset.reconstruct_output()에 넣을 수 있는 형태여야 함.
        """
        self.model = self.model.to(self.device)
        self.model.eval()

        # 1) dataset을 LIPS 스케일러로 변환(훈련때 fit한 스케일러 사용)
        test_loader = self.process_dataset(dataset=dataset, training=False)

        all_predictions = []

        # 평가 시 사용할 chunk 크기 (너의 PC RAM/GPU에 맞게 조절)
        # 너무 크면 메모리 터지고, 너무 작으면 느려짐
        chunk_size = int(kwargs.get("eval_chunk_size", 20000))
        k = int(self.hparams.get("k", 16))

        with torch.no_grad():
            for data in test_loader:   # batch_size=1, 한 번에 시뮬 1개
                data = data.to(self.device)

                N = data.x.size(0)
                pred_sim = torch.empty((N, 4), device="cpu")  # 결과는 CPU로 모아둠

                # 2) 전체 노드를 chunk로 나눠서 예측
                start = 0
                while start < N:
                    end = min(start + chunk_size, N)

                    # chunk 노드만 뽑기
                    pos_c = data.pos[start:end]
                    x_c   = data.x[start:end]

                    # chunk 내부에서만 kNN 그래프 생성 (평가용)
                    edge_index_c = build_knn_edge_index(pos_c, k=k, make_undirected=True).to(self.device)

                    # GAT 모델 입력 Data 구성
                    chunk_data = Data(pos=pos_c, x=x_c, edge_index=edge_index_c)
                    chunk_data = chunk_data.to(self.device)

                    out_c = self.model(chunk_data)          # (chunk,4)
                    out_c = out_c.detach().cpu()            # CPU로 이동

                    pred_sim[start:end] = out_c
                    start = end

                # 3) inverse_transform (정규화 복원)
                pred_sim_np = pred_sim.numpy()
                pred_sim_np = self._post_process(pred_sim_np)  # scaler inverse

                all_predictions.append(pred_sim_np)

        # 4) 시뮬레이션들을 위에서부터 쌓았으니 전체로 붙임
        predictions = np.vstack(all_predictions)

        predictions = predictions.astype(np.float32)

        # 5) LIPS가 기대하는 출력 구조로 복원
        predictions = dataset.reconstruct_output(predictions)
        return predictions
    
    
    def _post_process(self, data):
        try:
            processed = self.scaler.inverse_transform(data)
        except TypeError:
            processed = self.scaler.inverse_transform(data.cpu())
        return processed
    

    def train(self, train_dataset):
        base_loader = self.process_dataset(dataset=train_dataset, training=True)
        self.model = global_train(self.device, base_loader, self.model, self.hparams, criterion="MSE_weighted")
    
   


    

    def process_dataset(self, dataset, training: bool) -> DataLoader:
        coord_x = dataset.data['x-position']
        coord_y = dataset.data['y-position']
        surf_bool = dataset.extra_data['surface']
        position = np.stack([coord_x, coord_y], axis=1)

        nodes_features, node_labels = dataset.extract_data()


        if training:
            nodes_features, node_labels = self.scaler.fit_transform(nodes_features, node_labels)
        else:
            nodes_features, node_labels = self.scaler.transform(nodes_features, node_labels)

        torchDataset = []
        nb_nodes_in_simulations = dataset.get_simulations_sizes()
        start_index = 0

        for nb_nodes in nb_nodes_in_simulations:
            end_index = start_index + nb_nodes

            sim_pos = torch.tensor(position[start_index:end_index, :], dtype=torch.float)
            sim_x   = torch.tensor(nodes_features[start_index:end_index, :], dtype=torch.float)
            sim_y   = torch.tensor(node_labels[start_index:end_index, :], dtype=torch.float)
            sim_surf = torch.tensor(surf_bool[start_index:end_index]).bool()

            # 여기서는 edge_index를 만들지 않음! (노드가 너무 큼)
            # edge_index는 학습 시 subsampling 후에 생성함.
            sampleData = Data(pos=sim_pos, x=sim_x, y=sim_y, surf=sim_surf)
            torchDataset.append(sampleData)

            start_index = end_index

        return DataLoader(torchDataset, batch_size=1)


In [12]:
parameters = {
    "k": 12,
    "subsampling": 4096,
    "surf_ratio": 0.60,

    "hidden_dim": 128,
    "heads": 32,
    "dropout": 0.1,

    "batch_size": 1,
    "nb_epochs": 300,   # 처음엔 50으로 sanity check 추천
    "lr": 1e-3,

    "surf_reg": 3.0    # surface loss 가중치(필요하면 2~5로 올려보기)
}

mySimulator = AugmentedSimulator(benchmark=benchmark, **parameters)


Using cuda:0


In [13]:
mySimulator.train(benchmark.train_dataset)

[Epoch 10/300] loss=2.166274
[Epoch 20/300] loss=1.488452
[Epoch 30/300] loss=1.207595
[Epoch 40/300] loss=1.114381
[Epoch 50/300] loss=1.060162
[Epoch 60/300] loss=0.989858
[Epoch 70/300] loss=0.962557
[Epoch 80/300] loss=0.878792
[Epoch 90/300] loss=0.874177
[Epoch 100/300] loss=0.857318
[Epoch 110/300] loss=0.735097
[Epoch 120/300] loss=0.832992
[Epoch 130/300] loss=0.634556
[Epoch 140/300] loss=0.559748
[Epoch 150/300] loss=0.543652
[Epoch 160/300] loss=0.480107
[Epoch 170/300] loss=0.438971
[Epoch 180/300] loss=0.454470
[Epoch 190/300] loss=0.420118
[Epoch 200/300] loss=0.370970
[Epoch 210/300] loss=0.352939
[Epoch 220/300] loss=0.308812
[Epoch 230/300] loss=0.304844
[Epoch 240/300] loss=0.274767
[Epoch 250/300] loss=0.257638
[Epoch 260/300] loss=0.260965
[Epoch 270/300] loss=0.249403
[Epoch 280/300] loss=0.240008
[Epoch 290/300] loss=0.243230
[Epoch 300/300] loss=0.240493


In [18]:
start_test = time.time()
fc_metrics_test = benchmark.evaluate_simulator(dataset="test",augmented_simulator=mySimulator)
test_evaluation_time = time.time() - start_test
test_mean_simulation_time = test_evaluation_time/len(benchmark._test_dataset.get_simulations_sizes())
print("Test evaluation time:",test_evaluation_time)

print("Fully Connected Augmented Simulator")
ml_metrics = fc_metrics_test["test"]["ML"]
print("{:<10} : {}".format("Test ML metrics", ml_metrics))
physical_metrics = fc_metrics_test["test"]["Physics"]
print("{:<10} : {}".format("Test Physical metrics", physical_metrics))

start_test_ood = time.time()
fc_metrics_test_ood = benchmark.evaluate_simulator(dataset="test_ood",augmented_simulator=mySimulator, eval_batch_size=20000, num_workers=0)
test_ood_evaluation_time = time.time() - start_test_ood
test_ood_mean_simulation_time = test_ood_evaluation_time/len(benchmark._test_ood_dataset.get_simulations_sizes())
print("Test ood evaluation time:",test_ood_evaluation_time)

ood_ml_metrics = fc_metrics_test_ood["test_ood"]["ML"]
print("{:<10} : {}".format("OOD ML metrics", ood_ml_metrics))
ood_physical_metrics = fc_metrics_test_ood["test_ood"]["Physics"]
print("{:<10} : {}".format("OOD physical metrics", ood_physical_metrics))

Test evaluation time: 2407.389635324478
Fully Connected Augmented Simulator
Test ML metrics : {'MSE_normalized': {'x-velocity': 0.4572016831163147, 'y-velocity': 0.41335907853395343, 'pressure': 0.7913828184947532, 'turbulent_viscosity': 1.8458140391088151}, 'MSE_normalized_surfacic': {'pressure': 1.9710262304990682}}
Test Physical metrics : {'spearman_correlation_drag': 0.3098432460811521, 'spearman_correlation_lift': 0.9023505587639692, 'mean_relative_drag': 2.954112650567781, 'std_relative_drag': 3.2356235608130004, 'mean_relative_lift': 0.6357034413145016, 'std_relative_lift': 2.4249951845075244}
Test ood evaluation time: 5944.012395858765
OOD ML metrics : {'MSE_normalized': {'x-velocity': 0.5314467730715268, 'y-velocity': 0.5108415409539981, 'pressure': 1.048186631219689, 'turbulent_viscosity': 3.4630170535584965}, 'MSE_normalized_surfacic': {'pressure': 2.9459220280819385}}
OOD physical metrics : {'spearman_correlation_drag': 0.17495835226412237, 'spearman_correlation_lift': 0.86

In [19]:
def compute_global_score(test_result,ood_result,test_mean_simulation_time,test_ood_mean_simulation_time):
    thresholds={"x-velocity":(0.01,0.02,"min"),
            "y-velocity":(0.01,0.02,"min"),
            "pressure":(0.002,0.01,"min"),
            "pressure_surfacic":(0.008,0.02,"min"),
            "turbulent_viscosity":(0.05,0.1,"min"),
            "mean_relative_drag":(0.4,5.0,"min"),
            "mean_relative_lift":(0.1,0.3,"min"),
            "spearman_correlation_drag":(0.8,0.9,"max"),
            "spearman_correlation_lift":(0.96,0.99,"max")          
    }
    configuration={
        "coefficients":{"ML":0.4,"OOD":0.3,"Physics":0.3},
        "ratioRelevance":{"Speed-up":0.25,"Accuracy":0.75},
        "valueByColor":{"g":2,"o":1,"r":0},
        "maxSpeedRatioAllowed":10000,
        "reference_mean_simulation_time":1500
    }

    ml_metrics = test_result["test"]["ML"]["MSE_normalized"]
    ml_metrics["pressure_surfacic"] = test_result["test"]["ML"]["MSE_normalized_surfacic"]["pressure"]

    phy_variables_to_keep = ["mean_relative_drag","mean_relative_lift","spearman_correlation_drag","spearman_correlation_lift"]
    phy_metrics = {phy_variable:test_result["test"]["Physics"][phy_variable] for phy_variable in phy_variables_to_keep}

    ml_ood_metrics = ood_result["test_ood"]["ML"]["MSE_normalized"]
    ml_ood_metrics["pressure_surfacic"] = ood_result["test_ood"]["ML"]["MSE_normalized_surfacic"]["pressure"]
    phy_ood_metrics = {phy_variable:ood_result["test_ood"]["Physics"][phy_variable] for phy_variable in phy_variables_to_keep}
    ood_metrics = {**ml_ood_metrics,**phy_ood_metrics}

    all_metrics={
        "ML":ml_metrics,
        "Physics":phy_metrics,
        "OOD":ood_metrics
    }

    print(all_metrics)

    reference_mean_simulation_time=configuration["reference_mean_simulation_time"]
    speedUp={
        "ML":reference_mean_simulation_time/test_mean_simulation_time,
         "OOD":reference_mean_simulation_time/test_ood_mean_simulation_time
        }

    accuracyResults=dict()
    for subcategoryName, subcategoryVal in all_metrics.items():
        accuracyResults[subcategoryName]=[]
        for variableName, variableError in subcategoryVal.items():
            thresholdMin,thresholdMax,evalType=thresholds[variableName]
            if evalType=="min":
                if variableError<thresholdMin:
                    accuracyEval="g"
                elif thresholdMin<variableError<thresholdMax:
                    accuracyEval="o"
                else:
                    accuracyEval="r"
            elif evalType=="max":
                if variableError<thresholdMin:
                    accuracyEval="r"
                elif thresholdMin<variableError<thresholdMax:
                    accuracyEval="o"
                else:
                    accuracyEval="g"
    
            accuracyResults[subcategoryName].append(accuracyEval)

    print("accuracyResults: ",accuracyResults)
    
    coefficients=configuration["coefficients"]
    ratioRelevance=configuration["ratioRelevance"]
    valueByColor=configuration["valueByColor"]
    maxSpeedRatioAllowed=configuration["maxSpeedRatioAllowed"]
    mlSubscore=0
    accuracyMaxPoints=ratioRelevance["Accuracy"]
    accuracyResult=sum([valueByColor[color] for color in accuracyResults["ML"]])
    accuracyResult=accuracyResult*accuracyMaxPoints/(len(accuracyResults["ML"])*max(valueByColor.values()))
    mlSubscore+=accuracyResult
    print("ML accuracyResult",accuracyResult)

    speedUpMaxPoints=ratioRelevance["Speed-up"]
    speedUpResult=max(0,min(math.log10(speedUp["ML"])/math.log10(maxSpeedRatioAllowed),1))
    speedUpResult=speedUpResult*speedUpMaxPoints
    mlSubscore+=speedUpResult
    print("ML speedUpResult",speedUpResult)

    accuracyResult=sum([valueByColor[color] for color in accuracyResults["Physics"]])
    accuracyResult=accuracyResult/(len(accuracyResults["Physics"])*max(valueByColor.values()))
    physicsSubscore=accuracyResult
    print("Phy accuracyResult",accuracyResult)

    oodSubscore=0
    accuracyMaxPoints=ratioRelevance["Accuracy"]
    accuracyResult=sum([valueByColor[color] for color in accuracyResults["OOD"]])
    accuracyResult=accuracyResult*accuracyMaxPoints/(len(accuracyResults["OOD"])*max(valueByColor.values()))
    oodSubscore+=accuracyResult
    print("OOD accuracyResult",accuracyResult)

    speedUpMaxPoints=ratioRelevance["Speed-up"]
    speedUpResult=max(0,min(math.log10(speedUp["OOD"])/math.log10(maxSpeedRatioAllowed),1))
    speedUpResult=speedUpResult*speedUpMaxPoints
    oodSubscore+=speedUpResult
    print("OOD speedUpResult",speedUpResult)

    globalScore=100*(coefficients["ML"]*mlSubscore+coefficients["Physics"]*physicsSubscore+coefficients["OOD"]*oodSubscore)
    return globalScore

In [20]:
score = compute_global_score(fc_metrics_test,fc_metrics_test_ood,test_mean_simulation_time,test_ood_mean_simulation_time)
print(score)

{'ML': {'x-velocity': 0.4572016831163147, 'y-velocity': 0.41335907853395343, 'pressure': 0.7913828184947532, 'turbulent_viscosity': 1.8458140391088151, 'pressure_surfacic': 1.9710262304990682}, 'Physics': {'mean_relative_drag': 2.954112650567781, 'mean_relative_lift': 0.6357034413145016, 'spearman_correlation_drag': 0.3098432460811521, 'spearman_correlation_lift': 0.9023505587639692}, 'OOD': {'x-velocity': 0.5314467730715268, 'y-velocity': 0.5108415409539981, 'pressure': 1.048186631219689, 'turbulent_viscosity': 3.4630170535584965, 'pressure_surfacic': 2.9459220280819385, 'mean_relative_drag': 3.809186891792342, 'mean_relative_lift': 1.3055052932122817, 'spearman_correlation_drag': 0.17495835226412237, 'spearman_correlation_lift': 0.8606681142573919}}
accuracyResults:  {'ML': ['r', 'r', 'r', 'r', 'r'], 'Physics': ['o', 'r', 'r', 'r'], 'OOD': ['r', 'r', 'r', 'r', 'r', 'o', 'r', 'r', 'r']}
ML accuracyResult 0.0
ML speedUpResult 0.13097342926987465
Phy accuracyResult 0.125
OOD accuracyRes